# Web Deal Finder — Week 2 Exercise

**Learning goals (one per section):**
1. **Agent handoffs** — how one agent passes context to the next
2. **Structured `output_type`** — forcing an agent to return a typed Pydantic model
3. **`function_tool` side effects** — wrapping a real-world action (email) as a tool

**Pipeline:** `Scout` (web search) → `Comparator` (structured ranking) → `Publisher` (email)

**Cost:** Web search calls are billed by OpenAI — see [pricing](https://platform.openai.com/docs/pricing#web-search).

## Cell 1 — Imports and config

Nothing new here. `ModelSettings(tool_choice="auto")` lets the Scout decide when it has enough search evidence to stop — we don't force it to always call a tool.

In [24]:
!uv pip install markdown

Using Python 3.12.12 environment at: /Users/davidinyang-etoh/Projects/ai-projects/llm_agents/.venv
Audited 1 package in 18ms


In [25]:
import os
from typing import Dict

import sendgrid
from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.model_settings import ModelSettings
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv(override=True)

# ── configuration ────────────────────────────────────────────────────────────

MODEL           = "gpt-4o-mini"

FROM_EMAIL = "dainyangetoh@collectwire.com"
TO_EMAIL   = "davidinyangetoh@gmail.com"

## Cell 2 — Structured output schema (Learning goal: `output_type`)

When an agent has `output_type` set, the SDK forces it to return a valid instance of that model — not free-form text. The Comparator uses this.

Key discipline: **every field that might be unknown gets a literal `'unknown'` default**, not `None`. This prevents the model from hallucinating prices or URLs it didn't find in the search evidence.

In [26]:
class Deal(BaseModel):
    title:              str = Field(description="Product title grounded in search evidence")
    merchant_or_domain: str = Field(description="Store or domain name if found in evidence, else 'unknown'")
    price_text:         str = Field(description="Price as stated in evidence, else 'unknown'")
    url:                str = Field(description="URL only if present in evidence, else 'unknown'")
    why_ranked:         str = Field(description="One sentence tied to evidence — why this beat the others")


class DealFinderResult(BaseModel):
    ranked_deals:  list[Deal] = Field(description="Up to 3 best options, best first", max_length=3)
    alternatives:  list[str]  = Field(description="Close substitutes or different price tiers")
    prerequisites: list[str]  = Field(description="Accessories, compatibility notes, required subscriptions")
    caveats:       str        = Field(description="Shipping, region limits, price staleness warning")

## Cell 3 — Email tool (Learning goal: `function_tool` side effects)

A side-effect tool does something in the world — here it sends an email. A few things to notice:

1. The function is decorated with `@function_tool`. The SDK reads the docstring and type hints to build the JSON schema automatically.
2. **`SENDGRID_API_KEY` must be set** in `.env`; otherwise the tool raises — there is no dry-run path.
3. The Publisher passes **Markdown**; the tool converts it to HTML with a small **stdlib-only** helper (`html` + `re`) — no extra packages.
4. A footer disclaimer is appended before conversion — the model cannot omit it.

In [27]:
import markdown
from sendgrid.helpers.mail import Mail, Email, To, Content

@function_tool
def send_deal_summary_email(to_email: str, subject: str, body_markdown: str) -> Dict[str, str]:
    """Send deal summary: Markdown body is converted to HTML; disclaimer appended first. Requires SENDGRID_API_KEY."""
    disclaimer = "\n\n---\nPrices and availability are not guaranteed. Verify on the retailer site before purchasing."
    html_body = markdown.markdown(body_markdown + disclaimer)

    api_key = os.getenv("SENDGRID_API_KEY")
    if not api_key:
        raise ValueError("SENDGRID_API_KEY is not set. Add it to .env to send email.")
    
    sg = sendgrid.SendGridAPIClient(api_key=api_key)
    content = Content("text/html", html_body)
    mail = Mail(Email(FROM_EMAIL), To(to_email), subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "sent", "to": to_email}

## Cell 4 — Three agents and the handoff chain (Learning goal: agent handoffs)

A handoff is just one agent listing another in its `handoffs=[]`. When the running agent decides it's done, it transfers control and passes its output as context to the next agent.

The chain here is:
```
Scout  ──handoff──▶  Comparator  ──handoff──▶  Publisher
 (search)              (rank + structure)         (email)
```

**Why define agents bottom-up?** Each agent references the next in `handoffs=[]`. Python needs the target defined before it can be referenced, so Publisher is defined first.

In [28]:
# ── Publisher ─────────────────────────────────────────────────────────────────
# Receives the structured DealFinderResult, formats it, calls send_deal_summary_email once.
publisher_instructions = (
    f"You format the deal summary as clear Markdown (headings, bullet lists). "
    f"Call send_deal_summary_email exactly once with to_email={TO_EMAIL!r}, a short subject line, "
    "and body_markdown your summary. The tool converts Markdown to HTML and adds the legal disclaimer."
)

publisher_agent = Agent(
    name="Publisher",
    instructions=publisher_instructions,
    tools=[send_deal_summary_email],
    model=MODEL,
)

# ── Comparator ────────────────────────────────────────────────────────────────
# Receives Scout's notes, returns a DealFinderResult, then hands off to Publisher.
# output_type enforces structured output — the agent cannot reply with free-form text.
comparator_agent = Agent(
    name="Comparator",
    instructions=(
        "You receive web search notes from the Scout. "
        "Rank up to 3 deals using only prices, URLs, and merchants found in those notes. "
        "Use 'unknown' for anything not in the evidence — do not invent data. "
        "Return a DealFinderResult, then hand off to the Publisher."
    ),
    output_type=DealFinderResult,
    handoffs=[publisher_agent],
    model=MODEL,
)

# ── Scout ─────────────────────────────────────────────────────────────────────
# Entry point. Runs 2-4 web searches, collects evidence, hands off to Comparator.
scout_agent = Agent(
    name="Scout",
    instructions=(
        "You help shoppers find purchase options. "
        "Run 2 to 4 focused web searches (product name + buy + price; add region if given). "
        "Summarise what you found — prices, merchants, URLs from snippets only. "
        "Do not fabricate any data. When done, hand off to the Comparator with your notes."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model_settings=ModelSettings(tool_choice="auto"),
    handoffs=[comparator_agent],
    model=MODEL,
)

## Cell 5 — Run (live search)

`trace()` wraps the run so you can inspect every step at [platform.openai.com/traces](https://platform.openai.com/traces).

Change `product_ask` to whatever you want to shop for.

In [29]:
product_ask = "I need to buy a used 88 weighted keys digital piano in Nigeria, find the best price and merchant"

with trace("deal_finder"):
    result = await Runner.run(
        scout_agent,
        f"Shopping request: {product_ask}\nSend summary to: {TO_EMAIL}",
    )
display(Markdown(str(result.final_output)))

The summary of the used 88 weighted keys digital pianos has been successfully sent to your email at **davidinyangetoh@gmail.com**. If you need further assistance, feel free to ask!